# Attention Visualization

Hook into a transformer's attention layers and visualize attention weights for a given input. Useful for debugging what the model is actually attending to.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
class TinyAttention(nn.Module):
    def __init__(self, d=64, h=4):
        super().__init__()
        self.h = h; self.d = d; self.dh = d // h
        self.qkv = nn.Linear(d, 3 * d)
        self.out = nn.Linear(d, d)
        self.last_attn = None
    def forward(self, x):
        B, T, _ = x.shape
        qkv = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.h, self.dh).transpose(1, 2) for t in qkv]
        scores = (q @ k.transpose(-2, -1)) / self.dh ** 0.5
        attn = F.softmax(scores, dim=-1)
        self.last_attn = attn.detach()
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.out(out)

In [ ]:
torch.manual_seed(7)
T, d, h = 12, 64, 4
x = torch.randn(1, T, d)
attn = TinyAttention(d, h)
_ = attn(x)
fig, axes = plt.subplots(1, h, figsize=(14, 4))
for i in range(h):
    axes[i].imshow(attn.last_attn[0, i].numpy(), cmap='viridis')
    axes[i].set_title(f'head {i}')
    axes[i].set_xlabel('key pos')
    axes[i].set_ylabel('query pos')
fig.suptitle('Attention weights per head (random init)')
fig.tight_layout()
plt.show()

## Notes

- Random init produces uniform-looking attention; trained models show structure.
- Average attention across heads can be misleading -- different heads learn different patterns.
- For real transformers, hook on `attention_weights` outputs of the layer.